In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler

In [ ]:
# this ensures that the current MacOS version is at least 12.3+
print(torch.backends.mps.is_available())

# this ensures that the current current PyTorch installation was built with MPS activated.
print(torch.backends.mps.is_built())

## Loading Model

In [ ]:
# Paths are relative to the repo root - run this notebook from there.

# path to the model
output_dir = './FINAL_MODELS/model_CI_added_save_batch_16_1epochs/'

# Set datatype and device ("mps" instead of "cuda")
dtype = torch.float
device = torch.device("mps")

In [ ]:
# Load a trained model and vocabulary
model =  BertForSequenceClassification.from_pretrained(output_dir)
tokenizer = BertTokenizer.from_pretrained(output_dir)

# Copy the model to the GPU.
model.to(device)
device

## Run model on 'new' data

In [ ]:
# Load data
df = pd.read_csv("./scripts/pubmed_output/healthbase_articles_balanced_test_CI.csv")
df['CI_TiAb'] = df['CI_TiAb'].fillna('')
df.shape

In [ ]:
sum(df.case)

In [ ]:
len(df.case)

In [ ]:
df.head(2)

In [ ]:
# Report the number of sentences.
print('Number of test sentences: {:,}\n'.format(df.shape[0]))

# Create sentence and label lists.
# v2 uses CI_TiAb (contraindication + intervention + title + abstract), which is
# what this model was trained on — unlike v1, which uses TiAb (title + abstract).
sentences = list(df.CI_TiAb.values)
labels = list(df.case.to_numpy())

# Tokenize all of the sentences and map the tokens to their word IDs.
input_ids = []
attention_masks = []

# For every sentence...
for sent in sentences:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        sent,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        truncation = True,
                        max_length = 512,           # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )
    
    # Add the encoded sentence to the list.    
    input_ids.append(encoded_dict['input_ids'])
    
    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

# Set the batch size for DataLoader.  
batch_size = 64

# Create the DataLoader.
# prediction_data = TensorDataset(input_ids, attention_masks)
prediction_data = TensorDataset(input_ids, attention_masks, labels)
prediction_sampler = SequentialSampler(prediction_data)
prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=batch_size)

In [ ]:
# Prediction on set of articles
print('Predicting labels for {:,} found articles...'.format(len(input_ids)))

# Put model in evaluation mode
model.eval()

# Tracking variables 
predictions , true_labels = [], []
# predictions = []

# Predict 
for batch in prediction_dataloader:
  # Add batch to GPU
  batch = tuple(t.to(device) for t in batch) 
  
  # Unpack the inputs from our dataloader
  b_input_ids, b_input_mask, b_labels = batch
  # b_input_ids, b_input_mask = batch
  
  # Telling the model not to compute or store gradients, saving memory and 
  # speeding up prediction
  with torch.no_grad():
      # Forward pass, calculate logit predictions.
      result = model(b_input_ids, 
                     token_type_ids=None, 
                     attention_mask=b_input_mask,
                     return_dict=True)

  logits = result.logits

  # Move logits and labels to CPU
  logits = logits.detach().cpu().numpy()
  label_ids = b_labels.to('cpu').numpy()
  
  # Store predictions and true labels
  predictions.append(logits)
  true_labels.append(label_ids)

print('    DONE.')

In [ ]:
# Combine the results across all batches. 
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

# Calculate balanced accuracy
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score, precision_score, f1_score
balanced_accuracy_score(flat_true_labels, flat_predictions)

In [ ]:
# Combine the results across all batches. 
combined_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(combined_predictions, axis=1).flatten()

In [ ]:
import numpy as np

def softmax(x):
    """Compute softmax values for each sets of scores in x."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=1, keepdims=True)

# Convert logits to probabilities
probabilities = softmax(combined_predictions)

In [ ]:
probabilities

## Return dataframe with selected articles ranked by score

In [ ]:
# Return dataframe of all relevant articles with prediction score, ranked by highest score
relevant = df[flat_predictions.astype(bool)]

# Matching array of positive class ('1') prediction score
score = combined_predictions[:,1][flat_predictions.astype(bool)]

# Add score to dataframe
relevant['Score'] = score

# return dataframe of all articles with prediction score, ranked by highest score
relevant_sorted = relevant.sort_values(by='Score', ascending=False)

relevant_sorted.head(3)

## Check performance in this set

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, recall_score, roc_curve, roc_auc_score, f1_score
import matplotlib.pyplot as plt

# Combine the results across all batches. 
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

# Calculate metrics
print('Sensitivity: ', recall_score(flat_true_labels, flat_predictions))
print('Specificity: ', recall_score(flat_true_labels, flat_predictions, pos_label=0))
print('Balanced accuracy: ', balanced_accuracy_score(flat_true_labels, flat_predictions))
print('F1 score: ', f1_score(flat_true_labels, flat_predictions))

# Plot ROC-curve
predictions_1 = [pred[1] for pred in np.concatenate(predictions, axis=0)]

fpr, tpr, thresholds = roc_curve(flat_true_labels, predictions_1)
roc_auc = roc_auc_score(flat_true_labels, predictions_1)

# Determine the corresponding FPR values for the desired sensitivity levels
target_sensitivities = [0.95, 0.93, 0.90, 0.85]

selected_points = []

for ts in target_sensitivities:
    valid_indices = np.where(tpr >= ts)[0]

    if len(valid_indices) == 0:
        print(f"No threshold reaches sensitivity {ts:.2f}")
        continue

    # Highest specificity among thresholds that reach the target sensitivity
    best_idx = valid_indices[np.argmin(fpr[valid_indices])]

    selected_points.append({
        "target_sensitivity": ts,
        "actual_sensitivity": tpr[best_idx],
        "fpr": fpr[best_idx],
        "specificity": 1 - fpr[best_idx],
        "threshold": thresholds[best_idx],
    })

# Plot the ROC curve along with these points
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')

for point in selected_points:
    plt.plot(
        point["fpr"],
        point["actual_sensitivity"],
        'o',
        markersize=10,
        label=(
            f'Target Sens {point["target_sensitivity"]*100:.0f}% '
            f'(Actual {point["actual_sensitivity"]*100:.2f}%, '
            f'Spec {point["specificity"]*100:.2f}%)'
        )
    )

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# Calculate and print accuracy for target sensitivities
for point in selected_points:
    sensitivity = point["actual_sensitivity"]
    specificity = point["specificity"]
    cfpr = point["fpr"]

    TP = sensitivity * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    TN = specificity * np.sum(flat_true_labels == 0)
    FN = (1 - sensitivity) * np.sum(flat_true_labels == 1)

    accuracy = (TP + TN) / (TP + FP + TN + FN)
    balanced_acc = (sensitivity + specificity) / 2
    precision = TP / (TP + FP)
    f1_at_threshold = 2 * (precision * sensitivity) / (precision + sensitivity)

    print(f'Target Sensitivity {point["target_sensitivity"]*100:.0f}%')
    print(f'  Threshold: {point["threshold"]:.4f}')
    print(f'  Actual sensitivity: {sensitivity:.2f}')
    print(f'  Specificity: {specificity:.2f}')
    print(f'  Accuracy: {accuracy:.2f}')
    print(f'  Balanced accuracy: {balanced_acc:.2f}')
    print(f'  F1 score: {f1_at_threshold:.2f}')

In [ ]:
(sum(flat_predictions)-sum(flat_true_labels))/(len(flat_predictions)-sum(flat_true_labels))

In [ ]:
print(len(flat_predictions))
print(sum(flat_true_labels))
print(sum(flat_predictions))


In [ ]:
len(flat_predictions)/sum(flat_predictions)

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, recall_score, roc_curve, roc_auc_score, f1_score
import matplotlib.pyplot as plt

# Combine the results across all batches. 
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

# Calculate metrics
print('Sensitivity: ', recall_score(flat_true_labels, flat_predictions))
print('Specificity: ', recall_score(flat_true_labels, flat_predictions, pos_label=0))
print('Balanced accuracy: ', balanced_accuracy_score(flat_true_labels, flat_predictions))
print('F1 score: ', f1_score(flat_true_labels, flat_predictions))

# Plot ROC-curve
predictions_1 = [pred[1] for pred in np.concatenate(predictions, axis=0)]

fpr, tpr, thresholds = roc_curve(flat_true_labels, predictions_1)
roc_auc = roc_auc_score(flat_true_labels, predictions_1)

# Determine operating points for the desired sensitivity levels.
# An exact-match (atol) search can come up empty when the ROC has coarse TPR
# steps, which crashes np.argmin on an empty sequence. Instead take the point
# that achieves AT LEAST the target sensitivity at the best specificity.
target_sensitivities = [0.95, 0.93, 0.90, 0.85]
achieved_targets = []   # targets actually reachable on this curve
corresponding_fpr = []  # FPR at the chosen operating point
achieved_tpr = []       # actual TPR at the chosen point (>= target)

for ts in target_sensitivities:
    valid_indices = np.where(tpr >= ts)[0]
    if len(valid_indices) == 0:
        print(f'Skipping sensitivity {ts*100:.0f}%: not achievable (max TPR = {tpr.max():.3f})')
        continue
    best_idx = valid_indices[np.argmin(fpr[valid_indices])]
    achieved_targets.append(ts)
    corresponding_fpr.append(fpr[best_idx])
    achieved_tpr.append(tpr[best_idx])

# Plot the ROC curve along with these points
plt.figure(figsize=(8, 8))    
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')

# Plot the points for the target sensitivities (dot sits on the actual curve point)
for ts, cfpr, atpr in zip(achieved_targets, corresponding_fpr, achieved_tpr):
    plt.plot(cfpr, atpr, 'o', markersize=10,
             label=f'Sens>={ts*100:.0f}% (got {atpr*100:.1f}%, Spec {((1-cfpr)*100):.2f}%)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# Calculate and print accuracy for target sensitivities (using the actual
# achieved TPR/FPR at each chosen operating point).
for ts, cfpr, atpr in zip(achieved_targets, corresponding_fpr, achieved_tpr):
    TP = atpr * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    TN = (1 - cfpr) * np.sum(flat_true_labels == 0)
    FN = (1 - atpr) * np.sum(flat_true_labels == 1)
    
    accuracy = (TP + TN) / (TP + FP + TN + FN)
    print(f'Accuracy for Sensitivity {ts*100:.0f}%: {accuracy:.2f}')

# Calculate and print balanced accuracy for target sensitivities
for ts, cfpr, atpr in zip(achieved_targets, corresponding_fpr, achieved_tpr):
    specificity = 1 - cfpr
    balanced_acc = (atpr + specificity) / 2
    print(f'Balanced Accuracy for Sensitivity {ts*100:.0f}%: {balanced_acc:.2f}')


# Calculate and print F1 score for target sensitivities
for ts, cfpr, atpr in zip(achieved_targets, corresponding_fpr, achieved_tpr):
    TP = atpr * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    precision = TP / (TP + FP)
    f1 = 2 * (precision * atpr) / (precision + atpr)
    print(f'F1 Score for Sensitivity {ts*100:.0f}%: {f1:.2f}')


In [ ]:
# Function to convert logit to probability
def logit_to_prob(logit):
    return 1 / (1 + np.exp(-logit))

# Plot the ROC curve along with these points
plt.figure(figsize=(8, 8))    
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')

# Self-contained operating-point search (does not reuse lists from earlier cells).
# Same robust logic as the ROC cell above: take the point that achieves AT LEAST
# the target sensitivity at the best specificity, and skip unreachable targets.
target_sensitivities = [0.95]
achieved_targets = []
cutoffs_for_target_sens = []

for ts in target_sensitivities:
    valid_indices = np.where(tpr >= ts)[0]
    if len(valid_indices) == 0:
        print(f'Skipping sensitivity {ts*100:.0f}%: not achievable (max TPR = {tpr.max():.3f})')
        continue
    best_idx = valid_indices[np.argmin(fpr[valid_indices])]
    achieved_targets.append(ts)
    cutoffs_for_target_sens.append(thresholds[best_idx])

    plt.plot(fpr[best_idx], tpr[best_idx], 'o', markersize=10,
             label=f'Sens>={ts*100:.0f}% (got {tpr[best_idx]*100:.1f}%, Spec {((1-fpr[best_idx])*100):.2f}%)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# Print the cutoffs for each target sensitivity and their probabilities.
# NOTE: roc_curve thresholds here are raw positive-class logits, so logit_to_prob
# maps them to a probability. The final ROC point can carry threshold = +inf
# (sklearn's sentinel); logit_to_prob(inf) -> 1.0, which is expected.
for ts, cutoff in zip(achieved_targets, cutoffs_for_target_sens):
    prob_cutoff = logit_to_prob(cutoff)
    print(f'Cutoff for Sensitivity {ts*100:.0f}%: Logit = {cutoff:.2f}, Probability = {prob_cutoff:.2f}')

In [ ]:
def logit_to_prob(logit):
    return 1 / (1 + np.exp(-logit))

# Your logit cutoffs
logit_cutoffs = [-0.86, -0.69, -0.43, -0.20]

# Convert to probabilities
prob_cutoffs = [logit_to_prob(logit) for logit in logit_cutoffs]

# Print the probability cutoffs for each target sensitivity
for ts, prob in zip(target_sensitivities, prob_cutoffs):
    print(f'Probability Cutoff for Sensitivity {ts*100:.0f}%: {prob:.2f}')


In [ ]:
## AUROC with 95% CI (DeLong)
import sys
import numpy as np

# Absolute path to this project's scripts/ dir, so the import works regardless of
# the notebook's current working directory.
PROJECT_SCRIPTS = './scripts'
sys.path.append(PROJECT_SCRIPTS)
from delong import delong_ci

# Positive-class score per article, aligned to flat_true_labels.
# AUROC is rank-based, so raw logits and probabilities give the same value.
scores = np.concatenate(predictions, axis=0)[:, 1]
y_true = np.concatenate(true_labels, axis=0)

auc, (lo, hi) = delong_ci(y_true, scores, alpha=0.95)
print(f"V2 (CI_TiAb) AUROC = {auc:.3f} (95% CI {lo:.3f}-{hi:.3f})")
print(f"n = {len(y_true)}  |  positives = {int(y_true.sum())}  |  negatives = {int((y_true == 0).sum())}")